In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


In [1]:
PARAM <- list()
PARAM$semilla_primigenia <- 130079

PARAM$experimento <- 6307


In [2]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [3]:
require("data.table")

archivos <- c(
  "../WF6302/prediccion.txt",
  "../WF6303/prediccion.txt",
  "../WF6304/prediccion.txt",
  "../WF6305/prediccion.txt",
  "../WF6306/prediccion.txt"
)

# Lee cada archivo y los concatena en una sola tabla larga
tablas <- lapply(archivos, fread)
tb_predicciones <- rbindlist(tablas)

# Calcula el promedio agrupado por 'numero_de_cliente'
tb_prediccion <- tb_predicciones[, .(prob = mean(prob)), by = numero_de_cliente]

# Guarda el archivo consolidado del ensamble
fwrite(tb_prediccion,
  file = "prediccion_ensamblado.txt",
  sep = "\t"
)


Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




In [4]:

# Parámetros para Kaggle
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)


# Generación de envíos y Submit a Kaggle
# Ordena de mayor a menor probabilidad estimada
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0("./kaggle/KA_ensamblado", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}